# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [22]:
# =========================================
# Challenge 1: Combining & Cleaning Data
# =========================================

import pandas as pd
import numpy as np

# 1. Load datasets
file1 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
file2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv")
file3 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv")

# 2. Combine datasets
df = pd.concat([file1, file2, file3], ignore_index=True)

# 3. Standardize column names
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# 4. Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# 5. Clean text columns (only if present)
text_cols = ["gender", "education", "vehicle_class"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower()

# 6. Convert date column (only if present)
if "effective_to_date" in df.columns:
    df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")

# 7. Convert numeric columns (only if present)
numeric_cols = [
    "customer_lifetime_value",
    "income",
    "monthly_premium_auto",
    "total_claim_amount"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 8. Handle missing values
categorical_cols = df.select_dtypes(include="object").columns
df[categorical_cols] = df[categorical_cols].replace("nan", np.nan)
df[categorical_cols] = df[categorical_cols].fillna("unknown")

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# 9. Final check
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12074 entries, 0 to 12073
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   12074 non-null  object 
 1   st                         12074 non-null  object 
 2   gender                     12074 non-null  object 
 3   education                  12074 non-null  object 
 4   customer_lifetime_value    12074 non-null  float64
 5   income                     12074 non-null  float64
 6   monthly_premium_auto       12074 non-null  float64
 7   number_of_open_complaints  12074 non-null  object 
 8   policy_type                12074 non-null  object 
 9   vehicle_class              12074 non-null  object 
 10  total_claim_amount         12074 non-null  float64
 11  state                      12074 non-null  object 
dtypes: float64(4), object(8)
memory usage: 1.1+ MB


,customer,st,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount,state
0,RB50392,Washington,unknown,master,5782.095838,0.0,1000.0,1/0/00,Personal Auto,four-door car,2.704934,unknown
1,QZ44356,Arizona,f,bachelor,5782.095838,0.0,94.0,1/0/00,Personal Auto,four-door car,1131.464935,unknown
2,AI49188,Nevada,f,bachelor,5782.095838,48767.0,108.0,1/0/00,Personal Auto,two-door car,566.472247,unknown
3,WW63253,California,m,bachelor,5782.095838,0.0,106.0,1/0/00,Corporate Auto,suv,529.881344,unknown
4,GA49547,Washington,m,high school or below,5782.095838,36357.0,68.0,1/0/00,Personal Auto,four-door car,17.269323,unknown


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [23]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

df.head()


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [24]:
revenue_pivot = pd.pivot_table(
    df,
    values="total_claim_amount",
    index="sales_channel",
    aggfunc="sum"
).round(2)

revenue_pivot


,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [25]:
clv_pivot = pd.pivot_table(
    df,
    values="customer_lifetime_value",
    index="gender",
    columns="education",
    aggfunc="mean"
).round(2)

clv_pivot


education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [26]:
# Convert date column to datetime (safety check)
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")

# Extract month name
df["month"] = df["effective_to_date"].dt.month_name()


In [27]:
complaints_long = (
    df.groupby(["policy_type", "month"])["number_of_open_complaints"]
      .sum()
      .reset_index()
      .rename(columns={"number_of_open_complaints": "total_complaints"})
)

complaints_long


,policy_type,month,total_complaints
0,Corporate Auto,February,385.208135
1,Corporate Auto,January,443.434952
2,Personal Auto,February,1453.684441
3,Personal Auto,January,1727.605722
4,Special Auto,February,95.226817
5,Special Auto,January,87.074049
